# Week 3 Lab: Preprocessing, Pipelines & Data Leakage

AIE683 Machine Learning, Hacettepe University.

Worked examples follow the order of the slides:

1. scaling and pipelines on King County house sales (21,613 sales, May 2014 – May 2015),
2. `set_output`, `ColumnTransformer`, `make_column_selector`, infrequent categories,
3. target encoding of ZIP codes with `TargetEncoder` (cross-fitted),
4. power transforms and transforming the target,
5. four kinds of data leakage, each with a measured effect,
6. logging a leaky and a correct pipeline to MLflow.

Three **Try it** sections match the in-class activities. Their cells run as provided;
replace the `TODO` parts. All data is downloaded from OpenML at runtime.

In [1]:
import numpy as np
import pandas as pd
import mlflow

from sklearn.datasets import fetch_openml
from sklearn.model_selection import (train_test_split, cross_val_score, GridSearchCV,
                                     KFold, GroupKFold, TimeSeriesSplit)
from sklearn.preprocessing import (StandardScaler, MinMaxScaler, OneHotEncoder, TargetEncoder,
                                   PowerTransformer)
from sklearn.compose import (ColumnTransformer, make_column_transformer, make_column_selector,
                             TransformedTargetRegressor)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectPercentile, f_regression
from sklearn.linear_model import Ridge, RidgeCV, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

SEED = 0
cv = KFold(5, shuffle=True, random_state=SEED)

/Users/hemekci/Desktop/Hacettepe/AIE683-Machine-Learning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Scaling and pipelines

In [2]:
frame = fetch_openml("house_sales", version=1, as_frame=True).frame
target = frame.price
df = frame.drop(columns=["date", "price", "zipcode"])
X_train, X_test, y_train, y_test = train_test_split(df, target, random_state=SEED)
df.describe().T[["mean", "std", "min", "max"]].round(1)

,mean,std,min,max
bedrooms,3.4,0.9,0.0,33.0
bathrooms,2.1,0.8,0.0,8.0
sqft_living,2079.9,918.4,290.0,13540.0
sqft_lot,15107.0,41420.5,520.0,1651359.0
floors,1.5,0.5,1.0,3.5
waterfront,0.0,0.1,0.0,1.0
view,0.2,0.8,0.0,4.0
condition,3.4,0.7,1.0,5.0
grade,7.7,1.2,1.0,13.0
sqft_above,1788.4,828.1,290.0,9410.0


In [3]:
# Manual version: fit the scaler on the training set only, reuse it on the test set
scaler = StandardScaler().fit(X_train)
ridge = Ridge().fit(scaler.transform(X_train), y_train)
print("manual  :", round(ridge.score(scaler.transform(X_test), y_test), 3))

# The same thing as a pipeline
pipe = make_pipeline(StandardScaler(), Ridge()).fit(X_train, y_train)
print("pipeline:", round(pipe.score(X_test, y_test), 3))

manual  : 0.684
pipeline: 0.684


In [4]:
knn_pipe = make_pipeline(StandardScaler(), KNeighborsRegressor())
param_grid = {"kneighborsregressor__n_neighbors": [3, 5, 8, 12, 20]}
grid = GridSearchCV(knn_pipe, param_grid, cv=cv).fit(X_train, y_train)
print(grid.best_params_, round(grid.score(X_test, y_test), 3))

{'kneighborsregressor__n_neighbors': 5} 0.785


In [5]:
# set_output keeps pandas DataFrames (and column names) through transformers
StandardScaler().set_output(transform="pandas").fit(X_train).transform(X_test).iloc[:3, :5]

,bedrooms,bathrooms,sqft_living,sqft_lot,floors
17384,-1.448419,-0.801701,-0.704248,-0.321932,2.769831
722,0.667617,1.467506,2.799781,0.864530,0.925552
2680,-1.448419,-1.774219,-0.693433,-0.273468,-0.918726


## Try it 1: predict first, then run (10 minutes)

Three models on the house data, each **with** and **without** `StandardScaler`:
`KNeighborsRegressor()`, `RidgeCV()`, `HistGradientBoostingRegressor()`.

1. Before running anything, write down for each model whether scaling will help, hurt, or not matter.
2. Fill in `models` below (six pipelines) and run 5-fold CV on `X_train, y_train`.
3. Compare with your predictions. Which model families are scale-invariant, and why?

In [6]:
# TODO: add six entries, e.g. "kNN raw": KNeighborsRegressor(),
#       "kNN scaled": make_pipeline(StandardScaler(), KNeighborsRegressor()), ...
models = {
}

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv)
    print(f"{name:18s} R^2 = {scores.mean():.3f} +- {scores.std():.3f}")

## 2. Categorical variables and `ColumnTransformer`

In [7]:
boro = pd.DataFrame({
    "boro": ["Manhattan", "Queens", "Manhattan", "Brooklyn", "Brooklyn", "Bronx"],
    "salary": [103, 89, 142, 54, 63, 219],
    "vegan": ["No", "No", "No", "Yes", "Yes", "No"]})
pd.get_dummies(boro, columns=["boro"], dtype=int)

,salary,vegan,boro_Bronx,boro_Brooklyn,boro_Manhattan,boro_Queens
0,103,No,0,0,1,0
1,89,No,0,0,0,1
2,142,No,0,0,1,0
3,54,Yes,0,1,0,0
4,63,Yes,0,1,0,0
5,219,No,1,0,0,0


In [8]:
# A new batch with different categories gives different columns: get_dummies is not safe for train/test
boro_new = pd.DataFrame({"boro": ["Brooklyn", "Manhattan", "Queens", "Staten Island"],
                         "salary": [61, 146, 212, 47], "vegan": ["Yes", "No", "No", "No"]})
pd.get_dummies(boro_new, columns=["boro"], dtype=int).columns.tolist()

['salary',
 'vegan',
 'boro_Brooklyn',
 'boro_Manhattan',
 'boro_Queens',
 'boro_Staten Island']

In [9]:
# OneHotEncoder remembers the training categories; rare and unseen ones go to one column
ohe = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=2, sparse_output=False)
ohe.fit(boro[["boro"]])
print(ohe.get_feature_names_out(), ohe.infrequent_categories_)
ohe.transform(boro_new[["boro"]])

['boro_Brooklyn' 'boro_Manhattan' 'boro_infrequent_sklearn'] [array(['Bronx', 'Queens'], dtype=object)]


array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.],
       [0., 0., 1.]])

In [10]:
X_mixed, y_mixed = boro[["boro", "salary"]], boro.vegan
preprocess = ColumnTransformer([
    ("num", StandardScaler(), make_column_selector(dtype_include="number")),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
     make_column_selector(dtype_include=object)),
]).set_output(transform="pandas")
model = make_pipeline(preprocess, LogisticRegression()).fit(X_mixed, y_mixed)
preprocess.transform(X_mixed)

,num__salary,cat__boro_Bronx,cat__boro_Brooklyn,cat__boro_Manhattan,cat__boro_Queens
0,-0.155144,0.0,0.0,1.0,0.0
1,-0.405762,0.0,0.0,0.0,1.0
2,0.543005,0.0,0.0,1.0,0.0
3,-1.032307,0.0,1.0,0.0,0.0
4,-0.871196,0.0,1.0,0.0,0.0
5,1.921404,1.0,0.0,0.0,0.0


In [11]:
# Preprocessing choices are hyper-parameters: <step>__<transformer>__<parameter>
param_grid = {"columntransformer__cat__min_frequency": [None, 2],
              "logisticregression__C": [0.1, 1, 10]}
grid = GridSearchCV(model, param_grid, cv=2).fit(X_mixed, y_mixed)
grid.best_params_

{'columntransformer__cat__min_frequency': None, 'logisticregression__C': 1}

## 3. Target encoding: 70 ZIP codes

In [12]:
X_zip = frame.drop(columns=["date", "price"])
X_zip["zipcode"] = X_zip.zipcode.astype(str)
num_cols = make_column_selector(dtype_include="number")

pipe_nozip = make_pipeline(StandardScaler(), Ridge())
pipe_ohe = make_pipeline(make_column_transformer(
    (OneHotEncoder(handle_unknown="ignore"), ["zipcode"]),
    (StandardScaler(), num_cols)), Ridge())
pipe_te = make_pipeline(make_column_transformer(
    (TargetEncoder(target_type="continuous", cv=cv), ["zipcode"]),
    (StandardScaler(), num_cols)), Ridge())

print("no zipcode:", round(cross_val_score(pipe_nozip, df, target, cv=cv).mean(), 3))
print("one-hot:   ", round(cross_val_score(pipe_ohe, X_zip, target, cv=cv).mean(), 3))
print("target enc:", round(cross_val_score(pipe_te, X_zip, target, cv=cv).mean(), 3))

no zipcode: 0.693


one-hot:    0.805
target enc: 0.786


Note `target_type="continuous"`: prices such as `221900.0` are whole numbers, so automatic
type detection would treat the target as *multiclass*. Always set it explicitly for regression.

## 4. Power transforms and transforming the target

In [13]:
log_target = dict(func=np.log, inverse_func=np.exp)
for name, p in [
    ("StandardScaler", make_pipeline(StandardScaler(), RidgeCV())),
    ("PowerTransformer", make_pipeline(PowerTransformer(), RidgeCV())),
    ("PowerTransformer + log(y)", TransformedTargetRegressor(
        make_pipeline(PowerTransformer(), RidgeCV()), **log_target)),
]:
    scores = cross_val_score(p, X_train, y_train, cv=cv)
    print(f"{name:26s} R^2 = {scores.mean():.3f} +- {scores.std():.3f}")

StandardScaler             R^2 = 0.697 +- 0.011


PowerTransformer           R^2 = 0.617 +- 0.017


PowerTransformer + log(y)  R^2 = 0.717 +- 0.007


## 4b. Missing values and imputation

Real data has holes. Here we take iris and remove petal measurements on purpose, in a way that depends on the
values themselves (so *whether* a value is missing carries information). We compare dropping the incomplete
columns, median imputation, median imputation with missingness indicators, KNN imputation and iterative
(model-based) imputation. Every strategy sits **inside a pipeline**, so the imputer is fitted within each CV fold.


In [14]:
from sklearn.datasets import load_iris
from sklearn.utils import shuffle
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

iris = load_iris()
X, y = shuffle(iris.data, iris.target, random_state=0)
# values of petal length/width are removed where petal length is close to 5.5,
# which makes the missingness depend on the class (designed so mean imputation fails)
rng = np.random.RandomState(0)
X_ = X.copy()
mask = np.abs(X[:, 2] - rng.normal(loc=5.5, scale=.7, size=X.shape[0])) < .6
X_[mask, 3] = np.nan
mask2 = np.abs(X[:, 2] - rng.normal(loc=5.5, scale=.7, size=X.shape[0])) < .6
X_[mask2, 2] = np.nan
X_train, X_test, y_train, y_test = train_test_split(X_, y, stratify=y, random_state=0)
print("fraction of rows with a missing value:", np.isnan(X_train).any(axis=1).mean().round(3))

fraction of rows with a missing value: 0.375


In [15]:
# numbers quoted on the slides (10-fold CV accuracy on the training set)
nan_columns = np.any(np.isnan(X_train), axis=0)
pipes = {
    "drop columns": (make_pipeline(StandardScaler(), LogisticRegression()), X_train[:, ~nan_columns]),
    "median": (make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression()), X_train),
    "median + indicator": (make_pipeline(SimpleImputer(strategy='median', add_indicator=True),
                                         StandardScaler(), LogisticRegression()), X_train),
    "knn": (make_pipeline(KNNImputer(), StandardScaler(), LogisticRegression()), X_train),
    "iterative rf": (make_pipeline(IterativeImputer(estimator=RandomForestRegressor(n_estimators=20, random_state=0),
                                                    max_iter=5, random_state=0),
                                   StandardScaler(), LogisticRegression()), X_train),
}
for name, (pipe, data) in pipes.items():
    print(f"{name:20s} {cross_val_score(pipe, data, y_train, cv=10).mean():.3f}")

drop columns         0.830
median               0.830
median + indicator   0.883
knn                  0.920


/Users/hemekci/Desktop/Hacettepe/AIE683-Machine-Learning/.venv/lib/python3.12/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
/Users/hemekci/Desktop/Hacettepe/AIE683-Machine-Learning/.venv/lib/python3.12/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


/Users/hemekci/Desktop/Hacettepe/AIE683-Machine-Learning/.venv/lib/python3.12/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
/Users/hemekci/Desktop/Hacettepe/AIE683-Machine-Learning/.venv/lib/python3.12/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


/Users/hemekci/Desktop/Hacettepe/AIE683-Machine-Learning/.venv/lib/python3.12/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
/Users/hemekci/Desktop/Hacettepe/AIE683-Machine-Learning/.venv/lib/python3.12/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


/Users/hemekci/Desktop/Hacettepe/AIE683-Machine-Learning/.venv/lib/python3.12/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
/Users/hemekci/Desktop/Hacettepe/AIE683-Machine-Learning/.venv/lib/python3.12/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


iterative rf         0.938


/Users/hemekci/Desktop/Hacettepe/AIE683-Machine-Learning/.venv/lib/python3.12/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


Take-aways: missingness indicators (`add_indicator=True`) can help a lot when missingness is informative;
median + indicator is a strong, cheap default; tree ensembles in Week 5 (`HistGradientBoosting*`) handle `NaN` natively.


## 5. Data leakage

Kapoor & Narayanan (2023), *Leakage and the reproducibility crisis in machine-learning-based science*,
Patterns 4(9), 100804.

### 5a. Preprocessing fitted on the full data

Pure noise: 100 samples, 10,000 random features, random target. Nothing can be predicted.

In [16]:
rnd = np.random.RandomState(SEED)
X_noise, y_noise = rnd.normal(size=(100, 10000)), rnd.normal(size=100)
X_noise_test, y_noise_test = rnd.normal(size=(100, 10000)), rnd.normal(size=100)

# Leaky: select features using all 100 samples, then cross-validate
select = SelectPercentile(f_regression, percentile=5).fit(X_noise, y_noise)
X_selected = select.transform(X_noise)
leaky_cv = cross_val_score(Ridge(), X_selected, y_noise, cv=5).mean()
leaky_test = Ridge().fit(X_selected, y_noise).score(select.transform(X_noise_test), y_noise_test)

# Correct: selection is a pipeline step, refit inside every fold
pipe_sel = make_pipeline(SelectPercentile(f_regression, percentile=5), Ridge())
correct_cv = cross_val_score(pipe_sel, X_noise, y_noise, cv=5).mean()
correct_test = pipe_sel.fit(X_noise, y_noise).score(X_noise_test, y_noise_test)

print(f"leaky:    CV R^2 = {leaky_cv:.2f}, held-out R^2 = {leaky_test:.2f}")
print(f"pipeline: CV R^2 = {correct_cv:.2f}, held-out R^2 = {correct_test:.2f}")

leaky:    CV R^2 = 0.91, held-out R^2 = -0.10
pipeline: CV R^2 = -0.25, held-out R^2 = -0.10


### 5b. Target encoding without cross-fitting

`fit(X, y).transform(X)` encodes each training row with a category mean that contains its own target;
`fit_transform(X, y)` uses cross-fitting. A random categorical feature with 2,000 levels:

In [17]:
rng = np.random.RandomState(SEED)
X_cat = pd.DataFrame({"noise_id": rng.randint(0, 2000, size=5000).astype(str)})
y_cat = rng.normal(size=5000)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X_cat, y_cat, random_state=SEED)

te = TargetEncoder(target_type="continuous")
enc_leaky = te.fit(Xc_train, yc_train).transform(Xc_train)
enc_cross = te.fit_transform(Xc_train, yc_train)
enc_test = te.transform(Xc_test)
te_results = {}
for name, enc in [("fit().transform()", enc_leaky), ("fit_transform()", enc_cross)]:
    m = Ridge().fit(enc, yc_train)
    te_results[name] = (m.score(enc, yc_train), m.score(enc_test, yc_test))
    print(f"{name:18s} train R^2 = {te_results[name][0]:.3f}, test R^2 = {te_results[name][1]:.3f}")

fit().transform()  train R^2 = 0.456, test R^2 = -0.558
fit_transform()    train R^2 = 0.002, test R^2 = -0.001


### 5c. Target leakage: a feature computed from the target

In [18]:
df_leak = df.assign(price_per_sqft=target / df.sqft_living)
ridge_pipe = make_pipeline(StandardScaler(), RidgeCV())
print("honest features:", round(cross_val_score(ridge_pipe, df, target, cv=cv).mean(), 3))
print("with leak:      ", round(cross_val_score(ridge_pipe, df_leak, target, cv=cv).mean(), 3))

honest features: 0.693
with leak:       0.889


### 5d. Duplicates across folds (e.g. after oversampling before splitting)

In [19]:
sub = frame.sample(3000, random_state=SEED)
X_sub, y_sub = sub.drop(columns=["date", "price", "zipcode"]), sub.price
X_dup = pd.concat([X_sub] * 3, ignore_index=True)
y_dup = pd.concat([y_sub] * 3, ignore_index=True)
groups = np.tile(np.arange(len(X_sub)), 3)

rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=SEED)
print("original,   KFold:     ", round(cross_val_score(rf, X_sub, y_sub, cv=cv).mean(), 3))
print("duplicated, KFold:     ", round(cross_val_score(rf, X_dup, y_dup, cv=cv).mean(), 3))
print("duplicated, GroupKFold:", round(cross_val_score(rf, X_dup, y_dup, cv=GroupKFold(5),
                                                        groups=groups).mean(), 3))

original,   KFold:      0.831


duplicated, KFold:      0.984


duplicated, GroupKFold: 0.818


### 5e. Temporal leakage: training on the future

In [20]:
dates = pd.to_datetime(frame.date.str[:8], format="%Y%m%d")
order = np.argsort(dates.to_numpy(), kind="stable")
X_time, y_time = df.iloc[order], target.iloc[order]
hgb = HistGradientBoostingRegressor(random_state=SEED)
print("shuffled KFold: ", round(cross_val_score(hgb, X_time, y_time, cv=cv).mean(), 3))
print("TimeSeriesSplit:", round(cross_val_score(hgb, X_time, y_time, cv=TimeSeriesSplit(5)).mean(), 3))

shuffled KFold:  0.886


TimeSeriesSplit: 0.873


## 6. Logging the leaky and the correct pipeline to MLflow

Runs go to the experiment `week03-preprocessing` (local tracking store).
Browse them with `uv run mlflow ui` from this folder.

In [21]:
mlflow.set_experiment("week03-preprocessing")

for run_name, cv_r2, test_r2, protocol in [
        ("feature-selection / leaky", leaky_cv, leaky_test, "select on all data, then CV"),
        ("feature-selection / pipeline", correct_cv, correct_test, "Pipeline(select, ridge) inside CV")]:
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({"protocol": protocol, "percentile": 5, "n_samples": 100,
                           "n_features": 10000, "seed": SEED})
        mlflow.log_metrics({"cv_r2": cv_r2, "test_r2": test_r2, "optimism": cv_r2 - test_r2})
        mlflow.set_tag("leakage", "yes" if "leaky" in run_name else "no")

for run_name, key in [("target-encoding / fit then transform", "fit().transform()"),
                      ("target-encoding / cross-fitted", "fit_transform()")]:
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({"encoder": "TargetEncoder", "n_levels": 2000, "seed": SEED})
        mlflow.log_metrics({"train_r2": te_results[key][0], "test_r2": te_results[key][1]})
        mlflow.set_tag("leakage", "yes" if "fit then" in run_name else "no")

runs = mlflow.search_runs(experiment_names=["week03-preprocessing"],
                          order_by=["start_time DESC"], max_results=4)
runs[["tags.mlflow.runName", "tags.leakage", "metrics.cv_r2", "metrics.train_r2",
      "metrics.test_r2"]].round(3)

2026/09/30 00:00:34 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/09/30 00:00:34 INFO mlflow.store.db.utils: Updating database tables


2026/09/30 00:00:34 INFO mlflow.tracking.fluent: Experiment with name 'week03-preprocessing' does not exist. Creating a new experiment.


,tags.mlflow.runName,tags.leakage,metrics.cv_r2,metrics.train_r2,metrics.test_r2
0,target-encoding / cross-fitted,no,NaN,0.002,-0.001
1,target-encoding / fit then transform,yes,NaN,0.456,-0.558
2,feature-selection / pipeline,no,-0.247,NaN,-0.101
3,feature-selection / leaky,yes,0.906,NaN,-0.101


## Try it 2: Ames housing mini-competition (15 minutes)

The Ames housing data (1,460 houses, 79 features, 43 of them text columns, many missing values).
Target: log sale price; metric: RMSE on log price (lower is better).

1. Build **one pipeline** that handles numeric and categorical columns
   (`ColumnTransformer`, `make_column_selector`, `SimpleImputer`, `OneHotEncoder` with
   `handle_unknown="infrequent_if_exist"` and `min_frequency`, optionally `TargetEncoder` for `Neighborhood`).
2. Compare variants with 5-fold CV **on the training part only**.
3. Evaluate your final choice on the held-out test set **once** and post the number.

In [22]:
ames = fetch_openml("house_prices", as_frame=True)
X_ames = ames.data.drop(columns=["Id"])
y_ames = np.log(ames.target)
Xa_train, Xa_test, ya_train, ya_test = train_test_split(X_ames, y_ames, random_state=42)


def cv_rmse(model):
    scores = cross_val_score(model, Xa_train, ya_train, cv=cv,
                             scoring="neg_root_mean_squared_error")
    return -scores.mean()


# Baseline: numeric columns only
baseline = make_pipeline(
    make_column_transformer((SimpleImputer(strategy="median"),
                             make_column_selector(dtype_include="number"))),
    StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 20)))
print(f"baseline CV RMSE (log price): {cv_rmse(baseline):.4f}")

baseline CV RMSE (log price): 0.1572


In [23]:
# TODO: replace `my_model` with a pipeline that also uses the categorical columns
my_model = baseline

print(f"my model CV RMSE (log price): {cv_rmse(my_model):.4f}")

my model CV RMSE (log price): 0.1572


In [24]:
# Final evaluation: run this ONCE for the model you chose
final = my_model.fit(Xa_train, ya_train)
test_rmse = np.sqrt(np.mean((final.predict(Xa_test) - ya_test) ** 2))
print(f"held-out RMSE (log price): {test_rmse:.4f}")

held-out RMSE (log price): 0.1481


## Try it 3: find the leakage (15 minutes)

A colleague sends you the evaluation below and reports an impressive R². It contains
**several** leaks. With your neighbour:

1. List every leak you can find and say which of the four kinds from the slides it is.
2. Write a corrected evaluation (hint: one pipeline, a `TargetEncoder` inside it, and a
   splitter that respects both time and repeated houses).
3. Log both runs to MLflow (tag `leakage=yes/no`) and compare them in the MLflow UI.

In [25]:
# ---- colleague's code: do not edit, just read it ------------------------------------------
data = frame.copy()
data["sale_date"] = pd.to_datetime(data.date.str[:8], format="%Y%m%d")
data["zip_mean_price"] = data.groupby("zipcode").price.transform("mean")
data["price_per_sqft_zip"] = data.zip_mean_price / data.sqft_living
luxury = data[data.price > data.price.quantile(0.9)]
data = pd.concat([data] + [luxury] * 3, ignore_index=True)        # "balance" expensive houses

features = data.drop(columns=["date", "price", "sale_date", "zipcode"])
features = pd.DataFrame(StandardScaler().fit_transform(features), columns=features.columns)
colleague = HistGradientBoostingRegressor(random_state=SEED)
colleague_r2 = cross_val_score(colleague, features, data.price,
                               cv=KFold(5, shuffle=True, random_state=SEED)).mean()
print(f"colleague's CV R^2: {colleague_r2:.3f}")
# -------------------------------------------------------------------------------------------

colleague's CV R^2: 0.958


In [26]:
# TODO 1: list the leaks here (as comments)
#   -
#   -

# TODO 2: corrected evaluation. Start from `frame` (the original data), not from `data`.
fixed_r2 = None

# TODO 3: log both runs to MLflow
# with mlflow.start_run(run_name="find-the-leak / colleague"):
#     ...
print("fixed CV R^2:", fixed_r2)

fixed CV R^2: None
